In [2]:
# 기본 import
import os
import xml.etree.ElementTree as ET
import pandas as pd
from glob import glob

# 결절 정보 추출

In [3]:
def extract_nodules(xml_path):

		# xml 읽기
    tree = ET.parse(xml_path)
    root = tree.getroot()

		# 결과 저장 리스트 생성
    nodules = []

    #  XML 전체 순회
    for session in root.iter():

				# readingSession만 선택
        tag = session.tag.split("}")[-1]

        if tag != "readingSession":
            continue

        # session 내부 탐색
        for nodule in session:
						
						# 실제 결절만 선택
            nodule_tag = nodule.tag.split("}")[-1]

            if nodule_tag != "unblindedReadNodule":
                continue

						# 결절 정보 저장 dict
            info = {}

            # nodule 내부 탐색
            for child in nodule:

								# nodule ID 추출
                child_tag = child.tag.split("}")[-1]

                if child_tag == "noduleID":
                    info["nodule_id"] = child.text

            nodules.append(info)

    return nodules

In [4]:
# 폴더 경로 지정
folder = "/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157"

#폴더 안 파일 목록 가져오기
xml_file = os.listdir(folder)[0]

# 전체 경로 만들기 (folder + xml_file)
xml_path = os.path.join(folder, xml_file)

print(xml_path)

/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml/157/165.xml


In [5]:
# parsing 함수 실행
nodules = extract_nodules(xml_path)

print("nodule count:", len(nodules))
print(nodules[:5])

nodule count: 121
[{'nodule_id': '0'}, {'nodule_id': '2'}, {'nodule_id': '4'}, {'nodule_id': '5'}, {'nodule_id': '7'}]


# malignancy 추출

In [6]:
# xml 구조 확인
tree = ET.parse(xml_path)
root = tree.getroot()

for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "characteristics":

        for child in elem:

            print(child.tag, child.text)

        break

{http://www.nih.gov}subtlety 3
{http://www.nih.gov}internalStructure 1
{http://www.nih.gov}calcification 6
{http://www.nih.gov}sphericity 5
{http://www.nih.gov}margin 5
{http://www.nih.gov}lobulation 1
{http://www.nih.gov}spiculation 1
{http://www.nih.gov}texture 5
{http://www.nih.gov}malignancy 1


In [7]:
# 기존 함수에 characteristics 추가
def extract_nodules(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    nodules = []

    for session in root.iter():

        tag = session.tag.split("}")[-1]
        if tag != "readingSession":
            continue

        for nodule in session:

            nodule_tag = nodule.tag.split("}")[-1]
            if nodule_tag != "unblindedReadNodule":
                continue

            info = {}

            for child in nodule:

                child_tag = child.tag.split("}")[-1]

                # nodule ID
                if child_tag == "noduleID":
                    info["nodule_id"] = child.text

                # characteristics
                if child_tag == "characteristics":

                    for c in child:
                        ctag = c.tag.split("}")[-1]
                        info[ctag] = c.text

            # characteristics가 있는 경우만 추가
            if "malignancy" in info:
                nodules.append(info)

    return nodules

nodules = extract_nodules(xml_path)

print(len(nodules))
print(nodules[0])


21
{'nodule_id': 'CA006_12117', 'subtlety': '3', 'internalStructure': '1', 'calcification': '6', 'sphericity': '5', 'margin': '5', 'lobulation': '1', 'spiculation': '1', 'texture': '5', 'malignancy': '1'}


# roi_coords 추출

In [8]:
# XML 구조 확인
tree = ET.parse(xml_path)
root = tree.getroot()

for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "roi":

        print("FOUND ROI\n")

        for child in elem:
            print(child.tag)

        break

FOUND ROI

{http://www.nih.gov}imageZposition
{http://www.nih.gov}imageSOP_UID
{http://www.nih.gov}inclusion
{http://www.nih.gov}edgeMap


In [9]:
# edgeMap 구조 확인
for elem in root.iter():

    tag = elem.tag.split("}")[-1]

    if tag == "edgeMap":

        print("EDGE MAP")

        for child in elem:
            print(child.tag, child.text)

        break

EDGE MAP
{http://www.nih.gov}xCoord 306
{http://www.nih.gov}yCoord 255


In [10]:
# 기존 함수 안에 ROI 추가
def extract_nodules(xml_path):

    tree = ET.parse(xml_path)
    root = tree.getroot()

    nodules = []

    for session in root.iter():

        tag = session.tag.split("}")[-1]
        if tag != "readingSession":
            continue

        for nodule in session:

            nodule_tag = nodule.tag.split("}")[-1]
            if nodule_tag != "unblindedReadNodule":
                continue

            info = {}

            # ROI 저장용
            info["roi_list"] = []

            for child in nodule:

                child_tag = child.tag.split("}")[-1]

                # ROI 처리
                if child_tag == "roi":

                    roi_info = {}

                    coords = []

                    for roi_child in child:

                        roi_tag = roi_child.tag.split("}")[-1]

                        # z position
                        if roi_tag == "imageZposition":
                            roi_info["z"] = roi_child.text

                        # edge 좌표
                        if roi_tag == "edgeMap":

                            x = None
                            y = None

                            for edge_child in roi_child:

                                edge_tag = edge_child.tag.split("}")[-1]

                                if edge_tag == "xCoord":
                                    x = int(edge_child.text)

                                if edge_tag == "yCoord":
                                    y = int(edge_child.text)

                            if x is not None and y is not None:
                                coords.append((x, y))

                    roi_info["coords"] = coords

                    info["roi_list"].append(roi_info)

                # nodule ID
                if child_tag == "noduleID":
                    info["nodule_id"] = child.text

                # characteristics
                if child_tag == "characteristics":

                    for c in child:
                        ctag = c.tag.split("}")[-1]
                        info[ctag] = c.text

            # characteristics가 있는 경우만 추가
            if "malignancy" in info:
                nodules.append(info)

    return nodules

nodules = extract_nodules(xml_path)

print(nodules[0].keys())
print(nodules[0]["roi_list"][0])


dict_keys(['roi_list', 'nodule_id', 'subtlety', 'internalStructure', 'calcification', 'sphericity', 'margin', 'lobulation', 'spiculation', 'texture', 'malignancy'])
{'z': '-23.5', 'coords': [(307, 259), (308, 258), (308, 257), (308, 256), (309, 255), (309, 254), (308, 254), (307, 253), (306, 254), (305, 254), (304, 255), (304, 256), (304, 257), (304, 258), (305, 259), (306, 259), (307, 259)]}


# 전체 XML 데이터셋 구축

In [11]:
# 빈 리스트 생성
all_nodules = []

In [12]:
# 전체 xml 순회
xml_root = "/home/hdo/lidc_project/data/lidc-idri/LIDC-XML-only/tcia-lidc-xml"

for root_dir, dirs, files in os.walk(xml_root):

    for f in files:

        if not f.endswith(".xml"):
            continue

        xml_path = os.path.join(root_dir, f)

        try:
            nodules = extract_nodules(xml_path)

            # patient ID 추출
            patient_id = os.path.basename(root_dir)

            for nodule in nodules:

                nodule["patient_id"] = patient_id

                all_nodules.append(nodule)

        except Exception as e:

            print("ERROR:", xml_path)
            print(e)

In [13]:
# 데이터 확인
print(len(all_nodules))
print(all_nodules[0])

7032
{'roi_list': [{'z': '-191.699997', 'coords': [(194, 413), (195, 412), (196, 411), (196, 410), (196, 409), (196, 408), (196, 407), (196, 406), (196, 405), (195, 404), (194, 403), (193, 402), (192, 402), (191, 402), (190, 402), (189, 402), (188, 402), (187, 403), (186, 403), (185, 404), (185, 405), (185, 406), (185, 407), (185, 408), (185, 409), (185, 410), (186, 411), (186, 412), (187, 413), (188, 413), (189, 413), (190, 413), (191, 413), (192, 413), (193, 413), (194, 413)]}, {'z': '-192.949997', 'coords': [(191, 414), (192, 413), (193, 413), (194, 413), (195, 413), (196, 412), (197, 412), (197, 411), (198, 410), (198, 409), (198, 408), (198, 407), (198, 406), (198, 405), (198, 404), (197, 404), (196, 403), (196, 402), (195, 401), (194, 401), (193, 401), (192, 401), (191, 401), (190, 401), (189, 401), (188, 402), (187, 402), (186, 403), (185, 404), (184, 405), (184, 406), (184, 407), (184, 408), (184, 409), (184, 410), (184, 411), (185, 412), (186, 413), (187, 414), (188, 414), (18

In [14]:
# dataframe 생성
import pandas as pd

df = pd.DataFrame(all_nodules)

print(df.head())
print(df.columns)

                                            roi_list    nodule_id subtlety  \
0  [{'z': '-191.699997', 'coords': [(194, 413), (...         6272        5   
1  [{'z': '-192.949997', 'coords': [(190, 416), (...  IL057_69743        5   
2  [{'z': '-61.700001', 'coords': [(368, 326), (3...  IL057_69744        1   
3  [{'z': '-195.449997 ', 'coords': [(187, 400), ...   Nodule 001        5   
4  [{'z': '-191.699997', 'coords': [(193, 413), (...         4900        3   

  internalStructure calcification sphericity margin lobulation spiculation  \
0                 1             6          5      5          1           1   
1                 1             6          5      3          1           1   
2                 1             6          5      2          1           1   
3                 1             6          4      4          2           1   
4                 1             6          5      4          1           1   

  texture malignancy patient_id  
0       5          4        

# Label Cleaning

In [16]:
# 숫자 변환
df["malignancy"] = pd.to_numeric(
    df["malignancy"],
    errors="coerce"
)

# 분포 확인
df["malignancy"].value_counts().sort_index()

malignancy
0       2
1    1025
2    1654
3    2673
4     969
5     709
Name: count, dtype: int64

In [21]:
# ambiguous 제거
df = df[df["malignancy"] != 3]

# binary label 생성
df["label"] = df["malignancy"].apply(
    lambda x: 1 if x >= 4 else 0
)

print(df[["malignancy", "label"]].head())
print(df["label"].value_counts())

   malignancy  label
0           4      1
1           4      1
3           5      1
5           5      1
6           5      1
label
0    2681
1    1678
Name: count, dtype: int64
